# 02 — Data Cleaning & Preprocessing

This notebook takes the raw DWSIM Sobol batch output (`dataset_2560_converged.csv`) and prepares it for EDA and model training. The raw file already contains only the rows DWSIM marked as converged — non-convergent cases were logged separately (`dataset_2560_not_converged.csv`, 36 rows) and are not part of this pipeline.

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv(
    "../data/02_raw_train_sobol/dataset_2560_converged.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (2524, 25)


Shape check confirms we're starting from the full 2524-row converged batch, 25 columns as exported by the DWSIM automation script.

## Checking the Dataset Columns

In [3]:
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

 1. sobol_index
 2. batch
 3. pressure_atm
 4. requested_vapor_fraction
 5. benzene_feed_fraction
 6. toluene_feed_fraction
 7. stages
 8. feed_stage
 9. feed_stage_fraction
10. reflux_ratio
11. bottoms_fraction
12. feed_flow_kmol_h
13. feed_temperature_C
14. x_D_benzene
15. x_B_benzene
16. Q_C
17. Q_R
18. dwsim_solved
19. column_calculated
20. column_error
21. output_values_valid
22. composition_valid
23. temperature_valid
24. case_valid
25. error_message


## Removing Unnecessary Columns

`error_message` and `column_error` are diagnostic fields that only ever get populated when a DWSIM run fails to converge. Since this file has already been filtered to converged cases only, both columns are 100% null here by construction — they're not missing data in the usual sense, just fields that had nothing to report. They're dropped because they carry no information for this dataset and would otherwise show up as "missing values" in the checks below.

In [4]:
df_clean = df.drop(
    columns=[
        "error_message",
        "column_error"
    ]
)

print("Before:", df.shape)
print("After :", df_clean.shape)

Before: (2524, 25)
After : (2524, 23)


## Checking Data Types

In [5]:
df_clean.dtypes

sobol_index                   int64
batch                         int64
pressure_atm                float64
requested_vapor_fraction    float64
benzene_feed_fraction       float64
toluene_feed_fraction       float64
stages                        int64
feed_stage                    int64
feed_stage_fraction         float64
reflux_ratio                float64
bottoms_fraction            float64
feed_flow_kmol_h            float64
feed_temperature_C          float64
x_D_benzene                 float64
x_B_benzene                 float64
Q_C                         float64
Q_R                         float64
dwsim_solved                   bool
column_calculated              bool
output_values_valid            bool
composition_valid              bool
temperature_valid              bool
case_valid                     bool
dtype: object

## Checking Missing Values

In [6]:
df_clean.isnull().sum().sort_values(ascending=False)


sobol_index                 0
batch                       0
pressure_atm                0
requested_vapor_fraction    0
benzene_feed_fraction       0
toluene_feed_fraction       0
stages                      0
feed_stage                  0
feed_stage_fraction         0
reflux_ratio                0
bottoms_fraction            0
feed_flow_kmol_h            0
feed_temperature_C          0
x_D_benzene                 0
x_B_benzene                 0
Q_C                         0
Q_R                         0
dwsim_solved                0
column_calculated           0
output_values_valid         0
composition_valid           0
temperature_valid           0
case_valid                  0
dtype: int64

As expected, `error_message` and `column_error` account for all of the missing values here (2524 each = 5048 total), and every other column is fully populated. This matches: no run in this file failed, so there was nothing for those two columns to record.

## Checking Duplicate Samples

No duplicate rows is expected given the Sobol sequence used for sampling — it's a low-discrepancy, deterministic sequence, so two rows landing on the exact same 7-dimensional point is essentially impossible.

In [7]:
print(
    "Duplicate rows:",
    df_clean.duplicated().sum()
)

Duplicate rows: 0


## Checking NaN and Infinite Values

In [8]:
numeric_df = df_clean.select_dtypes(
    include=np.number
)

print(
    "NaN values:",
    numeric_df.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        numeric_df.to_numpy()
    ).sum()
)

NaN values: 0
Infinite values: 0


## Checking Input Ranges

These are the 6 of 7 sampled dimensions that have a fixed numeric range in the plan (`feed_stage_fraction` is checked separately below, since it interacts with `stages` — the actual feed stage is `round(fraction × stages)`, so a few rows can show a fraction slightly outside [0.30, 0.70] after rounding/clamping to a valid stage index).

In [9]:
ranges = {
    "pressure_atm": (1.0, 2.0),
    "requested_vapor_fraction": (0.0, 0.30),
    "benzene_feed_fraction": (0.30, 0.70),
    "stages": (10, 30),
    "reflux_ratio": (1.2, 4.5),
    "bottoms_fraction": (0.40, 0.60)
}

for column, (minimum, maximum) in ranges.items():

    outside = (
            (df_clean[column] < minimum) |
            (df_clean[column] > maximum)
    ).sum()

    print(
        f"{column}: {outside} values outside range"
    )

pressure_atm: 0 values outside range
requested_vapor_fraction: 0 values outside range
benzene_feed_fraction: 0 values outside range
stages: 0 values outside range
reflux_ratio: 0 values outside range
bottoms_fraction: 0 values outside range


## Checking Output Ranges

In [10]:
output_columns = [
    "feed_temperature_C",
    "x_D_benzene",
    "x_B_benzene",
    "Q_C",
    "Q_R"
]

df_clean[output_columns].describe()

,feed_temperature_C,x_D_benzene,x_B_benzene,Q_C,Q_R
count,2524.000000,2524.000000,2.524000e+03,2524.000000,2524.000000
mean,106.258194,0.883623,1.157872e-01,1616.301072,1628.816354
std,8.203032,0.134889,1.342727e-01,450.375487,450.943872
min,86.866737,0.510988,9.618306e-07,748.729567,753.118289
25%,99.966073,0.786738,3.354588e-03,1249.712729,1260.786887
50%,106.649918,0.951572,4.875739e-02,1588.163470,1600.171407
75%,112.577376,0.997273,2.119429e-01,1938.977047,1951.456410
max,124.212481,1.000000,4.938414e-01,2828.747577,2838.457528


## Checking Outliers

Using a standard IQR rule (1.5×IQR beyond Q1/Q3) here rather than removing anything automatically — the sampled inputs are bounded by design (Sobol/LHS within fixed ranges), so we don't expect statistical outliers on those. The outputs could in principle show real outliers from edge-case thermodynamic behavior, so it's worth checking even though none turned up.

In [11]:
outlier_columns = [
    "pressure_atm",
    "requested_vapor_fraction",
    "benzene_feed_fraction",
    "stages",
    "reflux_ratio",
    "bottoms_fraction",
    "feed_temperature_C",
    "x_D_benzene",
    "x_B_benzene",
    "Q_C",
    "Q_R"
]

for column in outlier_columns:

    Q1 = df_clean[column].quantile(0.25)
    Q3 = df_clean[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = (
            (df_clean[column] < lower) |
            (df_clean[column] > upper)
    ).sum()

    print(
        f"{column}: {count} outliers"
    )

pressure_atm: 0 outliers
requested_vapor_fraction: 0 outliers
benzene_feed_fraction: 0 outliers
stages: 0 outliers
reflux_ratio: 0 outliers
bottoms_fraction: 0 outliers
feed_temperature_C: 0 outliers
x_D_benzene: 0 outliers
x_B_benzene: 0 outliers
Q_C: 0 outliers
Q_R: 0 outliers


## Saving the Cleaned Dataset

Saving the cleaned frame (2524 rows × 23 columns, diagnostic-only columns dropped) for use in the EDA and model training notebooks.

In [12]:
processed_path = "../data/04_processed_dwsim"

os.makedirs(
    processed_path,
    exist_ok=True
)

cleaned_file = os.path.join(
    processed_path,
    "sobol_training_cleaned.csv"
)

df_clean.to_csv(
    cleaned_file,
    index=False
)

print("Saved:", cleaned_file)
print("Final shape:", df_clean.shape)
print(
    os.path.abspath(processed_path)
)

Saved: ../data/04_processed_dwsim\sobol_training_cleaned.csv
Final shape: (2524, 23)
C:\Users\Rupali Waghmare\Downloads\DWSIM_Task3_Surrogate_Model\data\04_processed_dwsim
